# 券商 App 新客开户转化链路 A/B 实验评估

本 notebook 使用 synthetic data 复现实验评估流程。

In [ ]:
import pandas as pd
assignment = pd.read_csv('../data/ab_assignment.csv', parse_dates=['assignment_time'])
events = pd.read_csv('../data/onboarding_events.csv', parse_dates=['event_time'])
post = pd.read_csv('../data/post_metrics.csv')
ab = assignment.merge(post, on='user_id')
ab.head()

In [ ]:
summary = ab.groupby('experiment_group').agg(users=('user_id','count'), completion_rate=('account_completed','mean'), retention_7d=('retention_7d','mean'), complaint_7d=('complaint_7d','mean')).reset_index()
summary

In [ ]:
funnel = events[events['status']=='success'].merge(assignment[['user_id','experiment_group']], on='user_id').groupby(['experiment_group','step_no','event_name'])['user_id'].nunique().reset_index(name='users').sort_values(['experiment_group','step_no'])
funnel['rate_from_start'] = funnel.groupby('experiment_group')['users'].transform(lambda s: s/s.iloc[0])
funnel

In [ ]:
channel_uplift = assignment.merge(post, on='user_id').groupby(['channel','experiment_group'])['account_completed'].mean().unstack().reset_index()
channel_uplift['uplift_ppt'] = (channel_uplift['treatment'] - channel_uplift['control']) * 100
channel_uplift.sort_values('uplift_ppt', ascending=False)

图表已预生成在 `../figures/`。